# rb2: la pregunta del comprador, con el proxy validado

Prerregistro en `docs/PREREG_rb2.md`. `rbuyer` no pudo contestar si `r512` acerca mas a la GT que
`r256` porque GT-en-pool dispara en ~20% de tareas (5 de 7 pares salieron 0-0). `strat` valido que
la **NLL/celda de la GT post-TTT** predice GT-en-pool (Spearman -0,430, separacion x11,6) e informa
en el 100% de las tareas.

Mismas 19 tareas estratificadas p5-p95 de `strat` (excluida la contaminada `36a08778`). GPUs 0-1
r=256/alpha=32 (campeon), GPUs 2-3 r=512/alpha=45 (rsLoRA a 2,0 identico). Sin decode. Misma
semilla md5 por tarea => mismas 128 secuencias de TTT en los dos brazos.

**PRERREG (n=19 pareado)**: ratio mediano <=0,85 y signo p<0,10 => HAY comprador, rama 2B viva.
>=1,15 con p<0,10 => mas rango aleja, rama muerta. Lo demas => empate = el rango no paga = no hay
comprador (negativo, registrado antes de ver el dato).


In [ ]:
%%writefile /kaggle/working/_worker_src.py
import glob, hashlib, json, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
T00 = time.time()

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "4"))
N_TAREAS = int(os.environ.get("N_TAREAS", "12"))
N_VISTAS = int(os.environ.get("N_VISTAS", "16"))
BUDGET = float(os.environ.get("BUDGET_VISTA", "45"))
WALL_MAX = float(os.environ.get("WALL_MAX", "2100"))
# DOS BRAZOS en GPUs separadas, un modelo por worker (cero fuga de estado).
# El lanzador fija BRAZO/R_LORA/ALPHA por entorno: 0-1 = r256/32 (campeon), 2-3 = r512/45
# (rsLoRA alpha/sqrt(r) = 2,0 en ambos). SIN decode: la primaria es la NLL post-TTT (proxy
# validado en strat: Spearman -0,430 con GT-en-pool, separacion x11,6).
BRAZO = os.environ.get("BRAZO", "r256")
R_LORA = int(os.environ.get("R_LORA", "256"))
ALPHA = int(os.environ.get("ALPHA", "32"))
SUCIAS = set(json.loads(os.environ["SUCIAS"]))
OUT = "/kaggle/working/shard%d.json" % SHARD

import numpy as np
import torch
from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict,
                  set_peft_model_state_dict)
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

# --- gate de carga: aborta ESTE shard, no la maquina ---
FICH = "arc-agi_evaluation_challenges.json"
DATA = None
for cand in ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
             "/kaggle/input/arc-prize-2026-arc-agi-2"):
    if os.path.exists(os.path.join(cand, FICH)):
        DATA = cand
        break
if DATA is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if FICH in files:
            DATA = root
            break
assert DATA, "sin dataset de la competicion"
CHAL = json.load(open(os.path.join(DATA, FICH)))
SOLS = json.load(open(os.path.join(DATA, "arc-agi_evaluation_solutions.json")))

MDIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and any(x.endswith(".safetensors") for x in files):
        MDIR = root
        break
assert MDIR, "no se encontro el modelo en /kaggle/input"
print("[s%d] modelo en %s" % (SHARD, MDIR), flush=True)

_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR = -float(np.log(0.2))
MAX_SEQ = 3500
LR = 5e-5


def coste(t):
    return sum(len(g["input"]) * len(g["input"][0]) + len(g["output"]) * len(g["output"][0])
               for g in t["train"])


# poblacion ESTRATIFICADA por coste a lo largo de TODA la distribucion: coger las N mas
# baratas fue el modo de fallo de lb33-cap (-2,5 pt). Indices equiespaciados en el ranking.
MIS_KEYS = json.loads(os.environ["MIS_KEYS"])
print("[s%d] brazo %s (r=%d alpha=%d) | %d tareas | costes %s"
      % (SHARD, BRAZO, R_LORA, ALPHA, len(MIS_KEYS), [coste(CHAL[k]) for k in MIS_KEYS]), flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def d4_inv(g, k):
    a = np.array(g)
    for _ in range(k % 4):
        a = np.rot90(a, -1)
    if k >= 4:
        a = a.T
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def invertir(g, k, perm):
    inv = [0] * 10
    for i, p in enumerate(perm):
        inv[p] = i
    return d4_inv([[inv[c] for c in fila] for fila in g], k)


def build_vistas(pares, seed, n):
    # vistas de DECODE: produccion usa augment(n=2) = 8 D4 x 2 perms = 16 vistas.
    rng = random.Random(seed)
    vistas = []
    for v in range(n):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = rng.sample(range(10), 10)
            orden = rng.sample(range(len(pares)), len(pares))
        vistas.append({"k": k, "perm": perm, "orden": orden})
    return vistas


def build_vistas_train(pares, seed, n_perm=16):
    # vistas de TTT: produccion usa augment(n=16) = 8 transformaciones D4 x 16
    # permutaciones de color = 128 SECUENCIAS, cada una vista UNA vez (1 epoca).
    # El PRODUCTO CRUZADO importa: 128 pasos, no 16. La primera version de este banco
    # generaba 16 vistas sueltas -> 8x menos adaptacion -> el 4B no aprendia nada
    # (3 de 16 tareas con GT en alguna vista). El smoke del 2B no lo detecto porque
    # ese modelo memoriza el eval y resuelve sin necesitar TTT.
    rng = random.Random(seed)
    perms = [list(range(10))] + [rng.sample(range(10), 10) for _ in range(n_perm - 1)]
    vistas = []
    for k in range(8):
        for perm in perms:
            vistas.append({"k": k, "perm": perm,
                           "orden": rng.sample(range(len(pares)), len(pares))})
    return vistas


base = AutoModelForCausalLM.from_pretrained(
    MDIR, torch_dtype=torch.bfloat16, device_map="cuda:0", attn_implementation="sdpa")
LCFG = LoraConfig(r=R_LORA, lora_alpha=ALPHA, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:
    OPT_CLS = torch.optim.AdamW
print("[s%d] modelo listo mem=%.1fGB (%.0fs)"
      % (SHARD, torch.cuda.memory_allocated() / 1e9, time.time() - T00), flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


@torch.no_grad()
def dfs(prompt, depth, budget):
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, score, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        nll = (score + torch.logsumexp(lf, dim=-1, keepdim=True)
               - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            sc = nll[0, j].item()
            if sc < THR:
                if t == EOS:
                    leaves.append((sc, list(path)))
                else:
                    cands.append((sc, t))
        for sc, t in sorted(cands):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], sc, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, cache0, depth)
    return leaves


@torch.no_grad()
def nll_de_gt(task, gt):
    """NLL de la rejilla correcta bajo el modelo YA ADAPTADO, vista canonica (k=0, perm
    identidad). Medida continua: mas potencia que el 0-16 con n=12. Es MECANISMO, no
    veredicto -- no confundir con la loss del TTT, que esta saturada."""
    prompt = []
    for p in task["train"]:
        t, _ = pair_tokens_mask(p["input"], p["output"])
        prompt += t
    prompt += query_tokens(task["test"][0]["input"])
    gtoks = grid_tokens(gt) + [EOS]
    ids = torch.tensor([prompt + gtoks], device="cuda:0", dtype=torch.long)
    with sdpa_kernel(_SDPA):
        out = MODEL(input_ids=ids, use_cache=False, return_dict=True)
    lg = out.logits[0, :-1].float()
    tgt = ids[0, 1:]
    lp = lg.log_softmax(-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    n_p = len(prompt)
    return float(-(lp[n_p - 1:]).sum().item()), len(gtoks)


def entrenar_ttt(task, vistas_train):
    # UNA epoca sobre las vistas: calendario EXACTO de produccion (128 secuencias vistas
    # una vez). Recortarlo es el modo de fallo de ttt05 (26,39) -- no se toca.
    MODEL.config.use_cache = False
    MODEL.train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    seqs = []
    for v in vistas_train:
        toks, mask = [], []
        for idx in v["orden"]:
            p = task["train"][idx]
            t, m = pair_tokens_mask(aplicar(p["input"], v["k"], v["perm"]),
                                    aplicar(p["output"], v["k"], v["perm"]))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            seqs.append((toks, mask))
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(len(seqs), 1))
    for toks, mask in seqs:
        ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
        lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                           device="cuda:0", dtype=torch.long)
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=ids, labels=lab)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in MODEL.parameters() if p.requires_grad], 1.0)
        opt.step()
        sch.step()
        opt.zero_grad(set_to_none=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    MODEL.eval()
    MODEL.config.use_cache = True
    return len(seqs)


RES = {"tareas": [], "modelo": "4b", "n_vistas": N_VISTAS, "budget_vista": BUDGET,
       "shard": SHARD, "brazo": BRAZO, "r": R_LORA, "alpha": ALPHA}
for i, k in enumerate(MIS_KEYS):
    if time.time() - T00 > WALL_MAX:
        print("[s%d] WALL_MAX alcanzado, corto en %d/%d" % (SHARD, i, len(MIS_KEYS)),
              flush=True)
        break
    task = CHAL[k]
    gt = SOLS[k][0]
    pares = [(p["input"], p["output"]) for p in task["train"]]
    seed = int(hashlib.md5(k.encode()).hexdigest()[:8], 16) % 1024**2

    set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
    try:
        _pre, _ = nll_de_gt(task, gt)
    except Exception:
        torch.cuda.empty_cache(); _pre = None
    v_train = build_vistas_train(pares, seed)
    assert len(v_train) == 128, "el TTT debe ver 128 secuencias como produccion"
    t0 = time.time()
    n_seqs = entrenar_ttt(task, v_train)
    t_ttt = time.time() - t0

    v_dec = build_vistas(pares, seed + 1, N_VISTAS)
    vistas_out, t0 = [], time.time()
    truncado = False
    for v in v_dec:
        if time.time() - T00 > WALL_MAX:          # deadline DENTRO de la tarea (leccion rbuyer)
            truncado = True
            print("[s%d] WALL_MAX dentro de %s: %d/%d vistas" % (SHARD, k, len(vistas_out), N_VISTAS), flush=True)
            break
        prompt = []
        for idx in v["orden"]:
            p = task["train"][idx]
            t, _ = pair_tokens_mask(aplicar(p["input"], v["k"], v["perm"]),
                                    aplicar(p["output"], v["k"], v["perm"]))
            prompt += t
        prompt += query_tokens(aplicar(task["test"][0]["input"], v["k"], v["perm"]))
        gt_v = grid_tokens(aplicar(gt, v["k"], v["perm"])) + [EOS]
        try:
            hojas = dfs(prompt, min(931, len(gt_v) + 40), BUDGET)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            hojas = []
        cands = []
        for sc, camino in sorted(hojas, key=lambda x: x[0])[:8]:
            g = toks_to_grid(camino)
            if g is None:
                continue
            try:
                cands.append({"grid": invertir(g, v["k"], v["perm"]), "score": round(sc, 4)})
            except Exception:
                continue
        vistas_out.append(cands)
        torch.cuda.empty_cache()
    t_dec = time.time() - t0

    n_con_gt = sum(1 for cs in vistas_out if any(c["grid"] == gt for c in cs))
    try:
        gt_nll, n_gt_tok = nll_de_gt(task, gt)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gt_nll, n_gt_tok = None, None
    except Exception as e:
        print("[s%d] nll_de_gt fallo en %s: %s" % (SHARD, k, str(e)[:120]), flush=True)
        gt_nll, n_gt_tok = None, None
    celdas = len(gt) * len(gt[0])
    RES["tareas"].append({"task": k, "brazo": BRAZO, "r": R_LORA, "vistas": vistas_out,
                          "n_seqs_ttt": n_seqs, "coste": coste(task), "celdas": celdas,
                          "ttt_s": round(t_ttt, 1), "decode_s": round(t_dec, 1),
                          "vistas_con_gt": n_con_gt,
                          "gt_nll": round(gt_nll, 3) if gt_nll is not None else None,
                          "gt_nll_celda": round(gt_nll / celdas, 4) if gt_nll is not None else None,
                          "gt_nll_celda_pre": round(_pre / celdas, 4) if _pre is not None else None,
                          "truncado": truncado, "vistas_hechas": len(vistas_out)})
    print("[s%d %d/%d] %s coste %d: GT en %d/%d vistas | nll/celda pre %s -> post %s | seqs %d | ttt %.0fs dec %.0fs%s"
          % (SHARD, i + 1, len(MIS_KEYS), k, coste(task), n_con_gt, len(vistas_out),
             ("%.4f" % (_pre / celdas)) if _pre is not None else "-",
             ("%.4f" % (gt_nll / celdas)) if gt_nll is not None else "-",
             n_seqs, t_ttt, t_dec, " TRUNCADO" if truncado else ""), flush=True)
    json.dump(RES, open(OUT, "w"))          # persistencia INCREMENTAL por tarea

json.dump(RES, open(OUT, "w"))
print("[s%d] fin %.0fs" % (SHARD, time.time() - T00), flush=True)


In [ ]:
import json, os, subprocess, sys, time
from math import comb
T0 = time.time()
SUCIAS = ["0934a4d8", "136b0064", "16b78196", "981571dc", "aa4ec2a5", "da515329"]
PLAN = [["faa9f03d", "21897d95", "3a25b0d8", "7b3084d4", "d59b0160", "4a21e3da", "67e490f4", "221dfab4", "195c6913"], ["28a6681f", "dd6b8c4b", "f931b4a8", "53fb4810", "b6f77b65", "58f5dbd5", "89565ca0", "cb2d8a2c", "a47bf94d", "b10624e5"], ["faa9f03d", "21897d95", "3a25b0d8", "7b3084d4", "d59b0160", "4a21e3da", "67e490f4", "221dfab4", "195c6913"], ["28a6681f", "dd6b8c4b", "f931b4a8", "53fb4810", "b6f77b65", "58f5dbd5", "89565ca0", "cb2d8a2c", "a47bf94d", "b10624e5"]]
BRAZOS = [("r256", 256, 32), ("r256", 256, 32), ("r512", 512, 45), ("r512", 512, 45)]

procs = []
for i in range(4):
    b, r, a = BRAZOS[i]
    env = dict(os.environ)
    env.update({"CUDA_VISIBLE_DEVICES": str(i), "SHARD": str(i), "N_SHARDS": "4",
                "BRAZO": b, "R_LORA": str(r), "ALPHA": str(a),
                "N_VISTAS": "0", "BUDGET_VISTA": "45", "WALL_MAX": "5400",
                "SUCIAS": json.dumps(SUCIAS), "MIS_KEYS": json.dumps(PLAN[i])})
    fh = open("/kaggle/working/log%d.txt" % i, "w")
    procs.append((i, subprocess.Popen([sys.executable, "/kaggle/working/_worker_src.py"],
                                      env=env, stdout=fh, stderr=subprocess.STDOUT), fh))
    print("worker %d: %s, %d tareas" % (i, b, len(PLAN[i])), flush=True)
for i, p, fh in procs:
    p.wait(); fh.close()
    print("worker %d rc=%d a los %.0fs" % (i, p.returncode, time.time() - T0), flush=True)

tareas = []
for i in range(4):
    f = "/kaggle/working/shard%d.json" % i
    if os.path.exists(f):
        tareas += json.load(open(f)).get("tareas", [])
json.dump({"tareas": tareas}, open("/kaggle/working/rb2.json", "w"))
for i in range(4):
    print("=" * 12 + (" LOG WORKER %d " % i) + "=" * 12, flush=True)
    try:
        print(open("/kaggle/working/log%d.txt" % i).read()[-2000:], flush=True)
    except Exception as e:
        print("sin log:", e, flush=True)

def med(xs):
    s = sorted(xs); n = len(s)
    return s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2.0
def signo(mas, menos):
    n = mas + menos
    if n == 0: return 1.0, 1.0
    k = max(mas, menos)
    dos = min(1.0, 2 * sum(comb(n, i) for i in range(k, n + 1)) / 2.0 ** n)
    una = sum(comb(n, i) for i in range(mas, n + 1)) / 2.0 ** n     # a favor de r512 (mas = ratio<1)
    return dos, una

por = {}
for t in tareas:
    por.setdefault(t["task"], {})[t["brazo"]] = t
pares = [(k, v["r256"], v["r512"]) for k, v in por.items()
         if "r256" in v and "r512" in v and v["r256"].get("gt_nll_celda") is not None and v["r512"].get("gt_nll_celda") is not None]
pares.sort(key=lambda x: x[1]["coste"])
print("")
print("=" * 88, flush=True)
print("rb2: r512 vs r256 SIN presion de reloj, primaria = NLL/celda de la GT post-TTT (proxy)", flush=True)
print("=" * 88, flush=True)
print("%-10s %6s | %9s %9s %7s | %9s %9s | %6s %6s" % ("tarea", "coste", "post r256", "post r512", "ratio", "pre r256", "pre r512", "s 256", "s 512"), flush=True)
ratios, mas, menos = [], 0, 0
for k, a, b in pares:
    r = b["gt_nll_celda"] / max(a["gt_nll_celda"], 1e-6)
    ratios.append(r); mas += r < 1; menos += r > 1
    print("%-10s %6d | %9.4f %9.4f %7.3f | %9s %9s | %6.0f %6.0f" % (
        k, a["coste"], a["gt_nll_celda"], b["gt_nll_celda"], r,
        ("%.4f" % a["gt_nll_celda_pre"]) if a.get("gt_nll_celda_pre") is not None else "-",
        ("%.4f" % b["gt_nll_celda_pre"]) if b.get("gt_nll_celda_pre") is not None else "-",
        a["ttt_s"], b["ttt_s"]), flush=True)
if pares:
    dos, una = signo(mas, menos)
    m = med(ratios)
    print("", flush=True)
    print("PRIMARIA ratio r512/r256: MEDIANA %.3f | media %.3f | r512 mejor en %d, peor en %d, empate %d | signo 2 colas p=%.4f, a favor de r512 p=%.4f"
          % (m, sum(ratios) / len(ratios), mas, menos, len(ratios) - mas - menos, dos, una), flush=True)
    pre_ok = sum(1 for _, a, b in pares if a.get("gt_nll_celda_pre") is not None and b.get("gt_nll_celda_pre") is not None
                 and abs(a["gt_nll_celda_pre"] - b["gt_nll_celda_pre"]) < 1e-3)
    print("CONTROL pre-TTT identico en ambos brazos: %d/%d (debe ser todos: es el modelo base)" % (pre_ok, len(pares)), flush=True)
    print("tiempo TTT total: r256 %.0fs -> r512 %.0fs = x%.2f (banco con grad-ckpt)" % (
        sum(a["ttt_s"] for _, a, _ in pares), sum(b["ttt_s"] for _, _, b in pares),
        sum(b["ttt_s"] for _, _, b in pares) / max(sum(a["ttt_s"] for _, a, _ in pares), 1)), flush=True)
    if m <= 0.85 and una < 0.10:
        ver = "HAY COMPRADOR: el rango acerca a la GT => rama Qwen3.5-2B VIVA"
    elif m >= 1.15 and (1 - una) < 0.10:
        ver = "NO HAY COMPRADOR: mas rango ALEJA (sobreajuste) => rama 2B MUERTA"
    else:
        ver = "EMPATE: el rango no paga => NO hay comprador => rama 2B BLOQUEADA (registrado como negativo)"
    print("", flush=True)
    print("VEREDICTO PRERREG: " + ver, flush=True)
    print("calibracion: +20% de NLL ~ -2,5 pt de LB (ancla gate-base)", flush=True)
else:
    print("SIN PARES", flush=True)
print("RESULT_JSON:" + json.dumps({"pares": len(pares), "wall_s": round(time.time() - T0)}), flush=True)
